In [1]:
%cd ..

/Users/v1adych/inno/data-mining/group_project


In [2]:
import polars as pl

events = pl.scan_parquet("data/events.parquet")

pr = events.filter(pl.col("event_type") == "PullRequestEvent").select(
    "event_id",
    "created_at",
    "actor_id",
    "actor_login",
    "repo_id",
    "repo_name",
    "payload_json",
)

pr_basic = pr.select(
    "event_id",
    "created_at",
    "actor_id",
    "actor_login",
    "repo_id",
    "repo_name",
    pl.col("payload_json").str.json_path_match("$.action").alias("action"),
    pl.col("payload_json").str.json_path_match("$.number").cast(pl.Int64).alias("pr_number"),
    pl.col("payload_json").str.json_path_match("$.pull_request.id").cast(pl.Int64).alias("pr_id"),
    pl.col("payload_json").str.json_path_match("$.pull_request.merged").alias("merged_raw"),
    pl.col("payload_json").str.json_path_match("$.pull_request.author_association").alias("author_association"),
    pl.col("payload_json").str.json_path_match("$.pull_request.draft").alias("draft_raw"),
    pl.col("payload_json").str.json_path_match("$.pull_request.commits").cast(pl.Int64).alias("commits"),
    pl.col("payload_json").str.json_path_match("$.pull_request.additions").cast(pl.Int64).alias("additions"),
    pl.col("payload_json").str.json_path_match("$.pull_request.deletions").cast(pl.Int64).alias("deletions"),
    pl.col("payload_json").str.json_path_match("$.pull_request.changed_files").cast(pl.Int64).alias("changed_files"),
)

pr_basic_df = pr_basic.collect()

pr_basic_df.shape

(470260, 16)

In [3]:
pr_basic_df.group_by("action").agg(pl.len().alias("count")).sort("count", descending=True)

action,count
str,u32
"""opened""",238541
"""closed""",228072
"""reopened""",3647


In [4]:
pr_basic_df.filter(pl.col("action") == "closed").group_by("merged_raw").agg(pl.len().alias("count")).sort("count", descending=True)

merged_raw,count
str,u32
"""true""",186811
"""false""",41261


In [5]:
pr_lifecycle_coverage = (
    pr_basic_df
    .filter(pl.col("action").is_in(["opened", "closed"]))
    .group_by("pr_id")
    .agg(
        pl.col("action").eq("opened").any().alias("has_opened"),
        pl.col("action").eq("closed").any().alias("has_closed"),
        pl.col("action").filter(pl.col("action") == "opened").len().alias("opened_events"),
        pl.col("action").filter(pl.col("action") == "closed").len().alias("closed_events"),
    )
)

pr_lifecycle_coverage.select(
    pl.len().alias("unique_prs"),
    (pl.col("has_opened") & pl.col("has_closed")).sum().alias("opened_and_closed"),
    pl.col("has_opened").sum().alias("has_opened"),
    pl.col("has_closed").sum().alias("has_closed"),
)

unique_prs,opened_and_closed,has_opened,has_closed
u32,u32,u32,u32
268072,195611,238541,225142
